## Travel time matrix generator
This notebook generates the travel time matrices of each damage scenario

In [ ]:
import wntr
import pandas as pd
import networkx as nx

In [13]:
# load the wdn as INP file
wn = wntr.network.WaterNetworkModel('BBM-EPS.inp') 

In [14]:
# function to create length of valves and pumps as 5.0 meters
def sim_length(i,pipes,wn):
    if i in pipes:
        return wn.get_link(i).length
    else:
        return 5.0

In [ ]:
# import the list of pipes of interest from the Excel file
file_rep = 'BPDRR_reparations.xlsx'
ds_sel = 'DS1'
file_distance = 'BBM_distances_' + ds_sel + '.csv'
FlagSave = 1
FlagPlot = 1

pipes_interest = pd.read_excel(
    file_rep,
    sheet_name=ds_sel,
    engine='openpyxl'
)

print(pipes_interest)

    Junction          Coefficient  Pipe ID
0              E1951      2.42800     1951
1              E3414      2.42800     3414
2              E4988      2.42800     4988
3              E5251      2.42800     5251
4              E3404      1.36575     3404
..               ...          ...      ...
101            E5959      0.29025     5959
102            E6041      0.29025     6041
103             E854      0.29025      854
104             E869      0.29025      869
105             E892      0.29025      892

[106 rows x 3 columns]


In [16]:
# List of pipe IDs
pipe_ids = pipes_interest["Pipe ID"].astype(str).tolist()

In [17]:
# Create a dictionary to store the start and end nodes for each pipe
pipe_nodes = {}

for pid in pipe_ids:

    link = wn.get_link(pid)

    pipe_nodes[pid] = {
        "start": link.start_node_name,
        "end": link.end_node_name
    }

In [18]:
# Graph where distances will be estimated
pipes  = wn.pipe_name_list
valves = wn.valve_name_list 
pumps  = wn.pump_name_list
links  = pipes + valves + pumps

start_node = [wn.get_link(i).start_node.name for i in pipes_interest['Pipe ID']]
final_node = [wn.get_link(i).end_node.name for i in pipes_interest['Pipe ID']]

l_dir = {(wn.get_link(i).start_node.name, 
          wn.get_link(i).end_node.name,
          sim_length(i,pipes,wn)) for i in links}

H = nx.Graph()
H.add_weighted_edges_from(l_dir)
H.number_of_nodes(), H.number_of_edges()

(4915, 6061)

In [19]:
# if loaded correctly then these values correspond to the number of reparations of the damage scenario on the excel file
len(start_node), len(final_node)

(106, 106)

In [20]:
# Create a distance matrix for the pipes of interest (not the nodes)
pipe_ids = list(pipe_nodes.keys())

distance_df = pd.DataFrame(
    index=pipe_ids,
    columns=pipe_ids,
    dtype=float
)

for p1 in pipe_ids:

    s1 = pipe_nodes[p1]["start"]
    e1 = pipe_nodes[p1]["end"]

    for p2 in pipe_ids:

        s2 = pipe_nodes[p2]["start"]
        e2 = pipe_nodes[p2]["end"]

        d1 = nx.shortest_path_length(H, s1, s2, weight="weight")
        d2 = nx.shortest_path_length(H, s1, e2, weight="weight")
        d3 = nx.shortest_path_length(H, e1, s2, weight="weight")
        d4 = nx.shortest_path_length(H, e1, e2, weight="weight")

        distance_df.loc[p1, p2] = (d1+d2+d3+d4)/4

In [21]:
# Convert meters to travel time in hours based on the speed of the crews
speed_kmh = 20.5
speed_mph = speed_kmh * 1000

travel_time = distance_df / speed_mph

In [22]:
# Save the travel time matrix to an Excel file
outfile = f"TravelTime_{ds_sel}.xlsx"

travel_time.to_excel(outfile)

print(f"Saved {outfile}")

Saved TravelTime_DS2.xlsx
